https://github.com/thor102/deliverable_3

Har brukt generativ AI til finpusse kode ut ifra utkast som vi har gitt til den. (Se sammendrag på canvas)

reference: <br>
orkslayergamedev, Python Classes Text Battle, GitHub, c. 2023, https://github.com/orkslayergamedev/python-classes-text-battle (accessed 03.10.2026).

Task 1: Create a Character class that allows characters to take damage, heal themselves, and check whether they are still alive.

In [2]:
class Character:
    """Represents a character in the RPG"""

    def __init__(self, name: str, hp_max: int = 100, damage: int = 10, weapon = None) -> None:
        """
        Initialise the Character

        Args:
            name (str): The name of the character.
            hp_max (int): Max hit points (default = 100).
            damage (int): 	Damage dealt when the character attacks (default = 10).
            weapon (Weapon): Optional weapon equipped by the character (default = None).
        """
        self.name = name
        self._hp = hp_max  # Note: _hp indicated private attribute
        self.hp_max = hp_max
        self.damage = damage
        self.weapon = weapon
        
    def attack(self, target) -> None:
        """Attack and give damage value to target character."""
        if self.weapon is not None:
            damage_done = self.weapon.roll_damage()
        else:
            damage_done = self.damage
        target.take_damage(damage_done)


    def take_damage(self, damage_done: int) -> None:
        """Takes damage from attack and takes it from hp amount."""
        self._hp -= abs(damage_done)
        self._hp = max(self._hp, 0)  # Target hp cannot end up lower than 0


    def heal(self, healing: int = 20) -> None:
        """Restore HP without exceeding the maximum HP."""
        self._hp += abs(healing) # abs makes sure that user can't use negtaive values.
        self._hp = min(self.hp_max, self._hp)

    @property
    def hp(self) -> int:
        """Public read-only property to inspect HP without allowing direct, unsafe edits."""
        return self._hp
    
    @property 
    def is_alive(self) -> bool:
        """Check whether the character is still alive."""
        if self.hp > 0:
            return True 
        else:
            return False 


In [3]:
# Test for task 1

hero = Character("Hero")
hero.take_damage(30)
hero.heal() # heals 20 hp
print("hero hp after damage and healing:", hero.hp)
print("is hero still alive?", hero.is_alive)
hero.take_damage(100) # Shows that .is_alive works
print("is hero alive?", hero.is_alive)


hero hp after damage and healing: 90
is hero still alive? True
is hero alive? False


**Task 1 Discussion** <br>
The Character class allows characters to receive damage and heal themselves. The HP value is kept between 0 and the maximum HP, and the is_alive property checks whether the character is still alive.

Task 2: The Dice class creates dice with a specified number of sides. If no number is provided, the default is 6 sides. The class can roll the dice and return a random number between 1 and the number of sides.

In [4]:
import random 


class Dice:
    """Represents a dice to make it more random what happens."""
    def __init__(self, sides: int = 6) -> None:
        """       
        Initialise the Dice
        Args:
            sides (int): The amount of sides on the dice (default 6)."""
        self.sides= max(sides, 2)


    def roll(self) -> int:
        """returns random value from 1 to 6 or the value the user sets."""
        return random.randint(1, self.sides)


    def __repr__(self) -> str:
        """Return a string representation of the Dice object."""
        return f"Dice(sides={self.sides})"     


    def __call__(self) -> int:
        """Allow the Dice object to be called directly to roll the dice."""
        return self.roll()        


In [5]:
# Test for task 2 
 
d20 = Dice(sides=20)
print(d20)

# Shows that both versions work 
print(d20.roll())
print(d20())


Dice(sides=20)
15
5


**Task 2 Discussion** <br>
The Dice class creates a dice with 6 sides by default, but the user can choose a different number of sides. The class also includes a __call__() method, which makes it easier to roll the dice by calling the object directly. 

Task 3: The Weapon class creates weapons that characters can use in battle. When a character has a weapon equipped, it automatically uses the weapon when attacking. The damage is determined by a random dice roll from the Dice class, instead of the character's default damage.

In [6]:
from dice import Dice


class Weapon:
    """Represents a weapon using dice for damage."""
    def __init__(self, name: str, damage_dice: Dice) -> None:  
        """              
        Initialise the Weapon
        Args:
            name (str): The name of the weapon. 
            damage_dice (Dice) The Dice object used to determine weapon damage."""
        self.name = name
        self.damage_dice = damage_dice 


    def roll_damage(self) -> int:
        """Return weapon damage based on a dice roll."""
        return self.damage_dice.roll() 


In [7]:
# Test for task 3

sword = Weapon("sword of the morning", Dice(20))
print(sword.roll_damage())


15


**Task 3 Discussion** <br>
One limitation of the Weapon class is that a weapon can deal less damage than an attack without a weapon. This could frustrate players and make weapons less useful. A possible improvement would be to add the weapon damage to the character's base damage instead of replacing it.

Task 4: The Character class is extended with two specialised subclasses, Warrior and Mage. Each subclass has its own abilities that affect how the character performs in battle.

In [8]:
class Warrior(Character):
    """A heavily armored character."""

    def __init__(self, name: str, armor: int = 5, **kwargs): 
        super().__init__(name=name, **kwargs)
        self.armor = armor

    def take_damage(self, damage_done: int) -> None:
        """Reduce incoming damage by armor."""
        reduced_damage = max(damage_done - self.armor, 0)
        super().take_damage(reduced_damage)

    def __str__(self) -> str:
        return (
            f"Warrior: {self.name}, "
            f"HP={self.hp}, "
            f"Damage={self.damage}, "
            f"Armor={self.armor}"
        )

class Mage(Character):
    """A spellcaster that uses mana."""

    def __init__(self, name: str, mana: int = 30, mana_max: int = 30, **kwargs) -> None:
        """
        Initialise the Mage.

        Args:
            name (str): The name of the character.
            mana (int): Starting mana (default = 30)
            mana_max (int): Max mana (default = 30)
            **kwargs: Passed on to Character (hp_max, damage, weapon, ...)
        """
        super().__init__(name=name, **kwargs)
        self.mana_max = mana_max
        self.mana = min(mana, mana_max)

    def attack(self, target) -> None:
        """
        Cast a spell (double damage, no weapon) if mana >= 10.
        Otherwise do a base attack (uses weapon if equipped) and restore 5 mana.
        """
        if self.mana >= 10:
            self.mana -= 10
            target.take_damage(2 * self.damage)
        else:
            super().attack(target)
            self.mana = min(self.mana + 5, self.mana_max)

    def __str__(self) -> str:
        return (
            f"Mage: {self.name}, HP={self.hp}, Damage={self.damage}, "
            f"Mana={self.mana}/{self.mana_max}"
        )



In [9]:
import inspect
import character
print(character.__file__)
print(inspect.signature(character.Mage.__init__))
print(open(character.__file__, encoding="utf-8").read().count("class Mage"))

c:\Users\madsh\Desktop\Inf201\Øvingsoppgaver\Uke 45-46\deliverable_3\character.py
(self, name: str, hp_max: int = 80, mana: int = 30, mana_max: int = 30, **kwargs) -> None
1


In [10]:
from character import Character, Warrior, Mage
from dice import Dice
from items import Weapon

# --- Warrior ---
w = Warrior("Bjorn")
assert w.armor == 5, "Warrior default armor should be 5"
assert w.hp == w.hp_max, "Warrior starts at full HP"

w = Warrior("Bjorn", hp_max=100)
w.take_damage(12)
assert w.hp == 93, "Armor should reduce 12 -> 7"
w.take_damage(2)
assert w.hp == 93, "Damage below armor should do 0 (not heal)"
w.take_damage(1000)
assert w.hp == 0 and not w.is_alive, "HP cannot go below 0"

w2 = Warrior("Tank", armor=3, hp_max=120, damage=15)
assert (w2.armor, w2.hp, w2.damage) == (3, 120, 15), "kwargs passed to Character"

# --- Mage: spell / fallback cycle ---
m = Mage("Merlin", damage=10)
assert (m.mana, m.mana_max) == (30, 30), "Mage default mana should be 30/30"
dummy = Character("Dummy", hp_max=1000)
for _ in range(3):
    m.attack(dummy)                      # 3 spells: 3 * 2 * 10 = 60
assert dummy.hp == 940 and m.mana == 0, "Spells cost 10 mana, double damage"
m.attack(dummy)                          # fallback: 10 dmg, +5 mana
assert dummy.hp == 930 and m.mana == 5, "Fallback restores 5 mana"
m.attack(dummy); m.attack(dummy)         # fallback (mana 10), then spell
assert dummy.hp == 900 and m.mana == 0

# --- Mage + weapon ---
staff = Weapon("Staff", Dice(sides=2))
t = Character("Target", hp_max=100)
Mage("A", damage=10, weapon=staff).attack(t)
assert t.hp == 80, "Spell ignores weapon"
t = Character("Target", hp_max=100)
m_low = Mage("B", mana=0, weapon=staff)
m_low.attack(t)
assert 98 <= t.hp <= 99 and m_low.mana == 5, "Fallback uses weapon"

assert Mage("C", mana=99).mana == 30, "Mana capped at mana_max"

# --- Polymorphism: spell goes through Warrior armor ---
w3 = Warrior("Shield", hp_max=100)
Mage("D", damage=10).attack(w3)
assert w3.hp == 85, "20 spell damage - 5 armor = 15"

assert isinstance(w3, Character) and isinstance(m, Character)
print(w3); print(m)
print("All Task 4 tests passed")

Warrior: Shield, HP=85, Damage=15, Armor=5
Mage: Merlin, HP=80, Damage=10, Mana=0/30
All Task 4 tests passed


Task 5: Create a Battle class that is running battles to one of the character have died.

In [11]:
from character import Character


class Battle:
    """Manage turn-based combat between two characters."""

    def __init__(self, hero: Character, enemy: Character, max_rounds: int = 1000) -> None:
        """
        Initialise the Battle.

        Args:
            hero (Character): Attacks first each round.
            enemy (Character): Attacks back if still alive.
            max_rounds (int): Safety limit to avoid infinite battles (default = 1000).
        """
        self.hero = hero
        self.enemy = enemy
        self.max_rounds = max_rounds

    def play_round(self) -> None:
        """Hero attacks enemy; if enemy survives, enemy attacks hero."""
        self.hero.attack(self.enemy)
        if self.enemy.is_alive:
            self.enemy.attack(self.hero)

    def simulate_battle(self) -> Character | None:
        """
        Play rounds until one character is no longer alive.

        Returns:
            Character | None: The winner, or None if max_rounds was reached.
        """
        round_number = 1
        while self.hero.is_alive and self.enemy.is_alive:
            if round_number > self.max_rounds:
                print("No winner: max rounds reached")
                return None
            print(f"\n--- Round {round_number} ---")
            self.play_round()
            print(f"{self.hero.name}: {self.hero.hp}/{self.hero.hp_max} HP")
            print(f"{self.enemy.name}: {self.enemy.hp}/{self.enemy.hp_max} HP")
            round_number += 1

        winner = self.hero if self.hero.is_alive else self.enemy
        print(f"\nWinner: {winner.name}")
        return winner

In [12]:
import random
from character import Character, Warrior, Mage
from dice import Dice
from items import Weapon
from battle import Battle

# --- play_round: both attack ---
hero, enemy = Character("Hero"), Character("Enemy")
Battle(hero, enemy).play_round()
assert hero.hp == 90 and enemy.hp == 90, "Both should attack once"

# --- play_round: dead enemy does not strike back ---
hero, enemy = Character("Hero"), Character("Enemy", hp_max=10)
Battle(hero, enemy).play_round()
assert not enemy.is_alive and hero.hp == 100, "Dead enemy must not attack"

# --- simulate_battle: correct winner and final HP ---
hero, enemy = Character("Hero"), Character("Enemy", hp_max=30)
winner = Battle(hero, enemy).simulate_battle()
assert winner is hero, "Hero should win"
assert enemy.hp == 0 and hero.hp == 80, "Enemy hits twice before dying"

# --- equal stats: hero wins because it attacks first ---
hero, enemy = Character("Hero"), Character("Enemy")
assert Battle(hero, enemy).simulate_battle() is hero

# --- stronger enemy wins ---
hero, enemy = Character("Hero"), Character("Boss", damage=50)
assert Battle(hero, enemy).simulate_battle() is enemy and hero.hp == 0

# --- subclasses and weapons ---
random.seed(1)
sword = Weapon("Sword", Dice(sides=12))
w, m = Warrior("Bjorn", weapon=sword), Mage("Merlin")
winner = Battle(w, m).simulate_battle()
assert winner in (w, m) and (w.is_alive != m.is_alive), "Exactly one survivor"

# --- max_rounds guard: 0 damage vs armor ---
a, b = Warrior("A", damage=3), Warrior("B", damage=3)
assert Battle(a, b, max_rounds=50).simulate_battle() is None

print("\nAll Task 5 tests passed")


--- Round 1 ---
Hero: 90/100 HP
Enemy: 20/30 HP

--- Round 2 ---
Hero: 80/100 HP
Enemy: 10/30 HP

--- Round 3 ---
Hero: 80/100 HP
Enemy: 0/30 HP

Winner: Hero

--- Round 1 ---
Hero: 90/100 HP
Enemy: 90/100 HP

--- Round 2 ---
Hero: 80/100 HP
Enemy: 80/100 HP

--- Round 3 ---
Hero: 70/100 HP
Enemy: 70/100 HP

--- Round 4 ---
Hero: 60/100 HP
Enemy: 60/100 HP

--- Round 5 ---
Hero: 50/100 HP
Enemy: 50/100 HP

--- Round 6 ---
Hero: 40/100 HP
Enemy: 40/100 HP

--- Round 7 ---
Hero: 30/100 HP
Enemy: 30/100 HP

--- Round 8 ---
Hero: 20/100 HP
Enemy: 20/100 HP

--- Round 9 ---
Hero: 10/100 HP
Enemy: 10/100 HP

--- Round 10 ---
Hero: 10/100 HP
Enemy: 0/100 HP

Winner: Hero

--- Round 1 ---
Hero: 50/100 HP
Boss: 90/100 HP

--- Round 2 ---
Hero: 0/100 HP
Boss: 80/100 HP

Winner: Boss

--- Round 1 ---
Bjorn: 105/120 HP
Merlin: 77/80 HP

--- Round 2 ---
Bjorn: 90/120 HP
Merlin: 67/80 HP

--- Round 3 ---
Bjorn: 75/120 HP
Merlin: 65/80 HP

--- Round 4 ---
Bjorn: 70/120 HP
Merlin: 60/80 HP

--- Round